This notebook was created by Donna Faith Go.

In [1]:
# standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import re

# pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear
from torch.utils.data import Dataset, DataLoader

# training
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import copy

# ignore warnings
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

## Accessing Data

In [2]:
# getting files
directory = "adjacency matrices 2"
files = sorted(
    f for f in os.listdir(directory) if f.endswith(".pkl")
)

# store information
dates = []
matrices = []

# get data
for f in files:
    with open(os.path.join(directory, f), "rb") as file:
        data = pickle.load(file)

    dates.append(data["date"])
    matrices.append(data["adjacency matrix"])

x = np.array(matrices)
print(x.shape)

(4063, 496, 496)


In [3]:
class StockShockDataset(Dataset):
    def __init__(self, adjacency_matrices, time_steps=10, negative_sampling_ratio=1.0):
        """
        Args:
            adjacency_matrices: numpy array of shape (n_periods, n_nodes, n_nodes)
            time_steps: number of time steps to use for each sample
            negative_sampling_ratio: ratio of negative to positive samples
        """
        self.adjacency_matrices = torch.FloatTensor(adjacency_matrices)
        self.n_periods, self.n_nodes, _ = adjacency_matrices.shape
        self.time_steps = min(time_steps, self.n_periods)
        self.negative_sampling_ratio = negative_sampling_ratio
        
        # Create node features (mean of returns - simplified as ones for now)
        # In practice, you'd want actual return data
        self.node_features = torch.ones(self.n_periods, self.n_nodes, 1)
        
    def __len__(self):
        return self.n_periods - self.time_steps
    
    def __getitem__(self, idx):
        # Get sequence of adjacency matrices
        adj_seq = self.adjacency_matrices[idx:idx + self.time_steps]
        
        # Get node features for the sequence
        x_seq = self.node_features[idx:idx + self.time_steps]
        
        # Get target adjacency matrix (next time step)
        target_adj = self.adjacency_matrices[idx + self.time_steps]
        
        # Sample positive edges (where target_adj == 1)
        pos_edges = torch.nonzero(target_adj > 0)
        
        # Sample negative edges (where target_adj == 0)
        neg_indices = torch.nonzero(target_adj == 0)
        n_neg = min(len(neg_indices), int(len(pos_edges) * self.negative_sampling_ratio))
        neg_idx = torch.randperm(len(neg_indices))[:n_neg]
        neg_edges = neg_indices[neg_idx]
        
        return x_seq, adj_seq, pos_edges, neg_edges

## Temporal Graph Convolutional Network

In [4]:
class GCNLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GCNLayer, self).__init__()
        self.linear = nn.Linear(in_channels, out_channels)
        
    def forward(self, x, adj):
        if adj.dim() == 2:
            adj = adj.unsqueeze(0).expand(x.size(0), -1, -1)
        support = self.linear(x)
        output = torch.bmm(adj, support)
        return output

In [5]:
class TGCNCell(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(TGCNCell, self).__init__()
        self.hidden_channels = hidden_channels
        self.in_channels = in_channels
        self.gcn_reset = GCNLayer(in_channels + hidden_channels, hidden_channels)
        self.gcn_update = GCNLayer(in_channels + hidden_channels, hidden_channels)
        self.gcn_candidate = GCNLayer(in_channels + hidden_channels, hidden_channels)
        
    def forward(self, x, adj, h):
        combined = torch.cat([x, h], dim=-1)
        r = torch.sigmoid(self.gcn_reset(combined, adj))
        z = torch.sigmoid(self.gcn_update(combined, adj))
        combined_reset = torch.cat([x, r * h], dim=-1)
        h_tilde = torch.tanh(self.gcn_candidate(combined_reset, adj))
        h_new = (1 - z) * h + z * h_tilde
        return h_new

In [6]:
class TGCNLayer(nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.3):
        super(TGCNLayer, self).__init__()
        self.tgcn_cell = TGCNCell(in_channels, hidden_channels)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.LeakyReLU()
        self.norm = nn.LayerNorm(hidden_channels)
        
    def forward(self, x_seq, adj):
        batch_size, time_steps, num_nodes, _ = x_seq.shape
        h = torch.zeros(batch_size, num_nodes, self.tgcn_cell.hidden_channels, device=x_seq.device)
        outputs = []
        for t in range(time_steps):
            h = self.tgcn_cell(x_seq[:, t, :, :], adj, h)
            h = self.activation(h)
            h = self.norm(h)
            h = self.dropout(h)
            outputs.append(h)
        return torch.stack(outputs, dim=1)

In [7]:
class TGCNModel(nn.Module):
    def __init__(self, num_nodes=498, in_channels=1, hidden_channels=128, 
                 num_layers=8, dropout=0.3, time_steps=10):
        super(TGCNModel, self).__init__()
        self.num_nodes = num_nodes
        self.time_steps = time_steps
        self.input_proj = nn.Linear(in_channels, hidden_channels)
        self.tgcn_layers = nn.ModuleList()
        for i in range(num_layers):
            layer_input_channels = hidden_channels if i > 0 else hidden_channels
            self.tgcn_layers.append(
                TGCNLayer(layer_input_channels, hidden_channels, dropout)
            )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.LeakyReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, 1)
        )
        self.reset_parameters()
        
    def reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
    
    def forward(self, x_seq, adj):
        batch_size = x_seq.size(0)
        x_seq = self.input_proj(x_seq)
        h = x_seq
        for i, layer in enumerate(self.tgcn_layers):
            h_new = layer(h, adj)
            if i > 0 and i % 2 == 1:
                if h.shape == h_new.shape:
                    h = h + h_new
                else:
                    h = h_new
            else:
                h = h_new
        final_hidden = h[:, -1, :, :]
        return final_hidden
    
    def predict_links(self, node_embeddings, edge_pairs):
        batch_size = node_embeddings.size(0)
        src_nodes = edge_pairs[:, 0]
        dst_nodes = edge_pairs[:, 1]
        src_emb = node_embeddings[:, src_nodes, :]
        dst_emb = node_embeddings[:, dst_nodes, :]
        edge_features = torch.cat([src_emb, dst_emb], dim=-1)
        logits = self.decoder(edge_features).squeeze(-1)
        return logits

## Training the TGCN

In [8]:
class TrainerWithEarlyStopping:
    def __init__(self, model, lr=0.001, weight_decay=1e-5, device='cuda', 
                 patience=10, min_delta=1e-4):
        self.model = model.to(device)
        self.device = device
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        self.criterion = nn.BCEWithLogitsLoss()
        self.patience = patience
        self.min_delta = min_delta
        self.best_model_state = None
        self.best_epoch = 0
        self.best_val_loss = float('inf')
        self.patience_counter = 0
        
    def train_epoch(self, train_loader, val_loader=None):
        self.model.train()
        total_loss = 0
        for batch in train_loader:
            x_seq, adj, pos_edges_list, neg_edges_list = batch
            
            # Process each item in the batch
            batch_loss = 0
            for i in range(len(x_seq)):
                x_seq_i = x_seq[i:i+1].to(self.device)
                adj_i = adj[i:i+1].to(self.device)
                pos_edges = pos_edges_list[i].to(self.device)
                neg_edges = neg_edges_list[i].to(self.device)
                
                node_embeddings = self.model(x_seq_i, adj_i)
                all_edges = torch.cat([pos_edges, neg_edges], dim=0)
                all_labels = torch.cat([
                    torch.ones(pos_edges.size(0)),
                    torch.zeros(neg_edges.size(0))
                ]).to(self.device)
                
                logits = self.model.predict_links(node_embeddings, all_edges)
                loss = self.criterion(logits, all_labels)
                batch_loss += loss
            
            self.optimizer.zero_grad()
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            total_loss += batch_loss.item()
        
        avg_loss = total_loss / len(train_loader)
        
        if val_loader is not None:
            val_loss, val_metrics = self.evaluate(val_loader, return_loss=True)
            return avg_loss, val_loss, val_metrics
        return avg_loss, None, None
    
    def evaluate(self, loader, return_loss=False):
        self.model.eval()
        all_logits = []
        all_labels = []
        total_loss = 0
        
        with torch.no_grad():
            for batch in loader:
                x_seq, adj, pos_edges_list, neg_edges_list = batch
                
                for i in range(len(x_seq)):
                    x_seq_i = x_seq[i:i+1].to(self.device)
                    adj_i = adj[i:i+1].to(self.device)
                    pos_edges = pos_edges_list[i].to(self.device)
                    neg_edges = neg_edges_list[i].to(self.device)
                    
                    node_embeddings = self.model(x_seq_i, adj_i)
                    all_edges = torch.cat([pos_edges, neg_edges], dim=0)
                    all_labels_batch = torch.cat([
                        torch.ones(pos_edges.size(0)),
                        torch.zeros(neg_edges.size(0))
                    ]).to(self.device)
                    
                    logits = self.model.predict_links(node_embeddings, all_edges)
                    
                    if return_loss:
                        loss = self.criterion(logits, all_labels_batch)
                        total_loss += loss.item()
                    
                    all_logits.append(logits.cpu())
                    all_labels.append(all_labels_batch.cpu())
        
        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels)
        probs = torch.sigmoid(all_logits)
        
        auc_roc = roc_auc_score(all_labels.numpy(), probs.numpy())
        auc_pr = average_precision_score(all_labels.numpy(), probs.numpy())
        preds = (probs > 0.5).float()
        f1 = f1_score(all_labels.numpy(), preds.numpy())
        
        metrics = {
            'auc_roc': auc_roc,
            'auc_pr': auc_pr,
            'f1': f1
        }
        
        if return_loss:
            return total_loss / len(loader), metrics
        return metrics
    
    def fit(self, train_loader, val_loader, n_epochs=150, verbose=True):
        train_losses = []
        val_losses = []
        val_metrics_history = []
        
        for epoch in range(n_epochs):
            train_loss, val_loss, val_metrics = self.train_epoch(train_loader, val_loader)
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            val_metrics_history.append(val_metrics)
            
            if val_loss < self.best_val_loss - self.min_delta:
                self.best_val_loss = val_loss
                self.best_epoch = epoch
                self.best_model_state = copy.deepcopy(self.model.state_dict())
                self.patience_counter = 0
                if verbose:
                    print(f"Epoch {epoch+1}: New best model! Val Loss: {val_loss:.4f}")
            else:
                self.patience_counter += 1
                if verbose and epoch % 10 == 0:
                    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, "
                          f"Val AUC-ROC: {val_metrics['auc_roc']:.4f}")
            
            if self.patience_counter >= self.patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}. Best epoch: {self.best_epoch+1}")
                break
        
        self.model.load_state_dict(self.best_model_state)
        
        return {
            'train_losses': train_losses,
            'val_losses': val_losses,
            'val_metrics': val_metrics_history,
            'best_epoch': self.best_epoch,
            'best_val_loss': self.best_val_loss
        }

In [9]:
def time_series_cv_evaluation(X, n_splits=5, time_steps=10, n_epochs=150, 
                              batch_size=16, device='cuda', verbose=True):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    cv_results = []
    
    print(f"Starting {n_splits}-fold Time Series Cross Validation")
    print("=" * 60)
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        print(f"\nFold {fold + 1}/{n_splits}")
        print("-" * 40)
        
        X_train, X_val = X[train_idx], X[val_idx]
        
        train_dataset = StockShockDataset(X_train, time_steps=time_steps)
        val_dataset = StockShockDataset(X_val, time_steps=time_steps)
        
        def custom_collate(batch):
            x_seq_list = []
            adj_list = []
            pos_edges_list = []
            neg_edges_list = []
            
            for item in batch:
                x_seq_list.append(item[0])
                adj_list.append(item[1])
                pos_edges_list.append(item[2])
                neg_edges_list.append(item[3])
            
            x_seq = torch.stack(x_seq_list)
            adj = torch.stack(adj_list)
            
            return x_seq, adj, pos_edges_list, neg_edges_list
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                                 shuffle=False, collate_fn=custom_collate)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, 
                               shuffle=False, collate_fn=custom_collate)
        
        model = TGCNModel(
            num_nodes=X.shape[1],
            in_channels=1,
            hidden_channels=128,
            num_layers=8,
            dropout=0.3,
            time_steps=time_steps
        )
        
        trainer = TrainerWithEarlyStopping(
            model, 
            lr=0.001, 
            weight_decay=1e-5, 
            device=device,
            patience=10
        )
        
        history = trainer.fit(train_loader, val_loader, n_epochs=n_epochs, verbose=verbose)
        
        final_metrics = trainer.evaluate(val_loader)
        final_metrics['best_epoch'] = history['best_epoch']
        final_metrics['best_val_loss'] = history['best_val_loss']
        
        cv_results.append(final_metrics)
        
        print(f"Fold {fold + 1} Results:")
        print(f"  AUC-ROC: {final_metrics['auc_roc']:.4f}")
        print(f"  AUC-PR: {final_metrics['auc_pr']:.4f}")
        print(f"  F1 Score: {final_metrics['f1']:.4f}")
        print(f"  Best Epoch: {history['best_epoch'] + 1}")
    
    print("\n" + "=" * 60)
    print("Cross-Validation Summary")
    print("=" * 60)
    
    summary = {}
    for metric in ['auc_roc', 'auc_pr', 'f1']:
        values = [r[metric] for r in cv_results]
        mean_val = np.mean(values)
        std_val = np.std(values)
        summary[metric] = {'mean': mean_val, 'std': std_val}
        print(f"{metric.upper()}: {mean_val:.4f} ± {std_val:.4f}")
    
    return cv_results, summary

In [10]:
def train_final_model(X, time_steps=10, n_epochs=150, batch_size=16, 
                      device='cuda', patience=15):
    print("\n" + "=" * 60)
    print("Training Final Model on All Data")
    print("=" * 60)
    
    split_idx = int(len(X) * 0.8)
    X_train, X_val = X[:split_idx], X[split_idx:]
    
    train_dataset = StockShockDataset(X_train, time_steps=time_steps)
    val_dataset = StockShockDataset(X_val, time_steps=time_steps)
    
    def custom_collate(batch):
        x_seq_list = []
        adj_list = []
        pos_edges_list = []
        neg_edges_list = []
        
        for item in batch:
            x_seq_list.append(item[0])
            adj_list.append(item[1])
            pos_edges_list.append(item[2])
            neg_edges_list.append(item[3])
        
        x_seq = torch.stack(x_seq_list)
        adj = torch.stack(adj_list)
        
        return x_seq, adj, pos_edges_list, neg_edges_list
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                             shuffle=False, collate_fn=custom_collate)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, 
                           shuffle=False, collate_fn=custom_collate)
    
    model = TGCNModel(
        num_nodes=X.shape[1],
        in_channels=1,
        hidden_channels=128,
        num_layers=8,
        dropout=0.3,
        time_steps=time_steps
    )
    
    trainer = TrainerWithEarlyStopping(
        model, 
        lr=0.001, 
        weight_decay=1e-5, 
        device=device,
        patience=patience
    )
    
    history = trainer.fit(train_loader, val_loader, n_epochs=n_epochs, verbose=True)
    
    final_metrics = trainer.evaluate(val_loader)
    
    print("\nFinal Model Results:")
    print(f"  AUC-ROC: {final_metrics['auc_roc']:.4f}")
    print(f"  AUC-PR: {final_metrics['auc_pr']:.4f}")
    print(f"  F1 Score: {final_metrics['f1']:.4f}")
    print(f"  Best Epoch: {history['best_epoch'] + 1}")
    
    return model, history, final_metrics

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

X = np.array(matrices)
print(f"Data shape: {X.shape}")

TIME_STEPS = 10
N_SPLITS = 5
BATCH_SIZE = 16
N_EPOCHS = 150
PATIENCE = 10

cv_results, cv_summary = time_series_cv_evaluation(
    X, 
    n_splits=N_SPLITS, 
    time_steps=TIME_STEPS,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    device=device,
    verbose=True
)

final_model, final_history, final_metrics = train_final_model(
    X,
    time_steps=TIME_STEPS,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    device=device,
    patience=PATIENCE
)

results = {
    'cv_results': cv_results,
    'cv_summary': cv_summary,
    'final_metrics': final_metrics,
    'final_history': final_history
}

with open('tgcn_training_results.pkl', 'wb') as f:
    pickle.dump(results, f)

torch.save(final_model.state_dict(), 'tgcn_final_model.pth')

print("\nTraining completed! Results saved to 'tgcn_training_results.pkl'")
print("Model saved to 'tgcn_final_model.pth'")

Using device: cpu
Data shape: (4063, 496, 496)
Starting 5-fold Time Series Cross Validation

Fold 1/5
----------------------------------------


RuntimeError: stack expects each tensor to be equal size, but got [119388, 2] at entry 0 and [128156, 2] at entry 1

## Supplementary

### Sanity Checks

In [ ]:
for filepath in sorted(os.listdir(r'adjacency matrices')):
    if filepath.endswith('.ipynb_checkpoints'):
        continue

    if filepath.endswith('01-11.pkl'):
        date_str = filepath
        filepath = f'adjacency matrices/{filepath}'
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        print(data)
        break
        df = pd.DataFrame(data['adjacency matrix'])
        
        # plotting
        plt.imshow(df)
        plt.title(f"{date_str}")
        plt.show()